# Streaming Checkpoints and Fault Tolerance: Exactly-Once Recovery

**Module**: 09_streaming  
**Notebook**: 05_checkpoints_and_fault_tolerance.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Checkpoint directory internals, offset write-ahead logs (WAL), state recovery, and achieving end-to-end exactly-once guarantees.
- **Why does this topic exist?**: Checkpoints allow streaming queries to restart after crashes from the exact point of interruption without data loss or duplication.
- **What problem does it solve?**: Stream restarting from scratch and duplicating side-effects on restart.
- **Where is it used in real-world Data Engineering?**: Zero-data-loss streaming pipelines.

---

## 2. Checkpoint Directory Structure: `offsets/`, `commits/`, `state/`, `metadata`

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Streaming_Checkpoint_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
chk_path = "/opt/spark-data/warehouse/chk_structure_demo"
os.system(f"rm -rf {chk_path}")

# Run short rate stream to generate checkpoint files
rate = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
q = rate.writeStream.format("noop").option("checkpointLocation", chk_path).start()
time.sleep(3)
q.stop()

print("Files created inside checkpoint directory:")
for root, dirs, files in os.walk(chk_path):
    for f in files:
        rel = os.path.relpath(os.path.join(root, f), chk_path)
        print(f"  - {rel}")

---

## Experiment: Resuming Stream from Checkpoint

- **1. What are we testing?**: Restart stream pointing to same checkpoint and verify it resumes from saved offset.
- **2. What do we expect?**: Stream continues from last committed micro-batch without reprocessing batch 0.

In [ ]:
q2 = rate.writeStream.format("noop").option("checkpointLocation", chk_path).start()
time.sleep(2)
last_id = q2.lastProgress["batchId"] if q2.lastProgress else 0
print(f"Resumed stream active! Last processed batch ID: {last_id}")
q2.stop()

**3. What happened?**: Stream resumed without re-reading historical batches.  
**4. Why did it happen?**: The commit log confirmed previous batches were already acknowledged.  
**5. Practical Engineering Lesson**: Never delete the checkpoint directory if you need uninterrupted recovery.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Commit Log
**Problem Statement**: Read the contents of the latest commit file in checkpoint.  
**Input / Context**: commits

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Exactly-Once Requirements
**Problem Statement**: List the 4 requirements for exactly-once streaming.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Changing Query Logic
**Problem Statement**: Can you change DataFrame query logic with an existing checkpoint?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Commit Log):
```python
print('Commit file records batch completion status.')
```

#### Solution 2 (Exactly-Once Requirements):
```python
print('1. Replayable source, 2. Deterministic execution, 3. Idempotent sink, 4. Checkpointing.')
```

#### Solution 3 (Changing Query Logic):
```python
print('Certain breaking schema changes or stateful operator mutations require starting a new checkpoint directory.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Exactly-Once**: Achieved through offset log + idempotent sink.
- **Checkpoint Integrity**: Must be stored on reliable storage.

### Essential Technical Terms
- `WAL`: Write-ahead log recording offsets.
- `Commit Log`: Confirms batch completion.

### Common Engineering Mistakes
- **Storing checkpoints on local /tmp**: Container restart wipes checkpoints, causing stream to reprocess from beginning.

### Interview Questions on This Topic
- **Q: What are the components of a streaming checkpoint directory?**
  - *A*: offsets/ (uncommitted source offsets), commits/ (completed batches), state/ (state store snapshots), and metadata (unique query run ID).

### What to Study Next
Open 06_kafka_integration.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')